# Sample ARC Submission

This is a sample notebook that can help get you started with creating an ARC Prize submission. It covers the basics of loading libraries, loading data, implementing an approach, and submitting.

You should be able to submit this notebook to the evaluation portal and have it run successfully (although you'll get a score of 0, so you'll need to do some work if you want to do better!)


# Load needed libraries

Basic libraries like numpy, torch, matplotlib, and tqdm are already installed.

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm


# Load the data

Here we are loading the training challenges and solutions (this is the public training set), the evaluation challenges and solutions (this is the public evaluation set), and the test challenges (currently a placeholder file that is a copy of the public evaluation challanges).

For your initial testing and exploration, I'd recommend not using the public evaluation set, just work off the public training set and then test against the test challenges (which is actually the public evaluation set). However, when competing in the competition, then you can should probably use the evaluation tasks for training too.

Note, currently `../input/problems` contains the ARC-AGI-1 problem set, but you can copy the contents of `../input/ARC-AGI-1/` or `../input/ARC-AGI-2/` into this folder, if you'd like to play around with testing on both ARC-AGI-1 and ARC-AGI-2 problem sets.

In [ ]:
# Public training set, public evaluation set, and hidden test set.
# The starter notebook is usually run from the working/ directory, but Gradescope-like
# runners may place the notebook in a slightly different directory. Try the common
# layouts before failing.
def find_problem_dir():
    candidates = [
        Path('../input/problems'),
        Path('input/problems'),
        Path('/kaggle/input/problems'),
        Path('/mnt/data/input/problems'),
    ]
    for candidate in candidates:
        if (candidate / 'arc-agi_training_challenges.json').exists() and (candidate / 'arc-agi_test_challenges.json').exists():
            return candidate
    raise FileNotFoundError('Could not find ARC problem files. Expected an input/problems directory.')


def load_json_if_present(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path) as fp:
        return json.load(fp)


problem_dir = find_problem_dir()

train_challenges_path = problem_dir / 'arc-agi_training_challenges.json'
train_solutions_path = problem_dir / 'arc-agi_training_solutions.json'
evaluation_challenges_path = problem_dir / 'arc-agi_evaluation_challenges.json'
evaluation_solutions_path = problem_dir / 'arc-agi_evaluation_solutions.json'
test_challenges_path = problem_dir / 'arc-agi_test_challenges.json'

with open(train_challenges_path) as fp:
    train_challenges = json.load(fp)
train_solutions = load_json_if_present(train_solutions_path, {})

evaluation_challenges = load_json_if_present(evaluation_challenges_path, {})
evaluation_solutions = load_json_if_present(evaluation_solutions_path, {})

with open(test_challenges_path) as fp:
    test_challenges = json.load(fp)

print(f'Loaded ARC data from {problem_dir}')
print(f'Training tasks: {len(train_challenges)}; test tasks: {len(test_challenges)}')



Here is an example of what a test task looks like:

In [ ]:
sample_task = list(test_challenges.keys())[0]
print('Sample task id:', sample_task)
print('Training examples:', len(test_challenges[sample_task]['train']))
print('Test examples:', len(test_challenges[sample_task]['test']))


# Generating a submission

To generate a submission you need to output a file called `submission.json` that has the following format:

```
{"00576224": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 "009d5c81": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 "12997ef3": [{"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]},
              {"attempt_1": [[0, 0], [0, 0]], "attempt_2": [[0, 0], [0, 0]]}],
 ...
}
```

In this case, the task ids come from `test_challenges`. There may be multiple (i.e., >1) test items per task. Therefore, the dictionary has a list of dicts for each task. These submission dictionaries should appear in the same order as the test items from `test_challenges`. Additionally, you can provide two attempts for each test item. In fact, you **MUST** provide two attempts. If you only want to generate a single attempt, then just submit the same answer for both attempts (or submit an empty submission like the ones shown in the example snippit just above.

Here is how we might create a blank submission:

# Similarity-Difference-Color solver

The solver below follows a simple knowledge-based strategy: compare similarities, isolate differences, and then apply color mappings. It searches a small library of explicit transformation rules and keeps only rules that reproduce every training output for a task.



In [ ]:
from collections import Counter, deque


def grid_shape(grid):
    return (len(grid), len(grid[0]) if grid else 0)


def copy_grid(grid):
    return [row[:] for row in grid]


def grids_equal(a, b):
    return a == b


def most_common_color(grid):
    counts = Counter(cell for row in grid for cell in row)
    return counts.most_common(1)[0][0] if counts else 0


def colors_in(grid):
    return set(cell for row in grid for cell in row)


def crop_grid(grid, top, left, height, width):
    return [row[left:left + width] for row in grid[top:top + height]]


def bounding_box(points):
    if not points:
        return None
    rows = [r for r, _ in points]
    cols = [c for _, c in points]
    return min(rows), min(cols), max(rows) + 1, max(cols) + 1


def crop_bbox(grid, bbox):
    if bbox is None:
        return None
    top, left, bottom, right = bbox
    return crop_grid(grid, top, left, bottom - top, right - left)


def non_background_bbox(grid):
    bg = most_common_color(grid)
    points = [(r, c) for r, row in enumerate(grid) for c, value in enumerate(row) if value != bg]
    return bounding_box(points)


def color_bbox(grid, color):
    points = [(r, c) for r, row in enumerate(grid) for c, value in enumerate(row) if value == color]
    return bounding_box(points)


def connected_components(grid, include_background=False):
    height, width = grid_shape(grid)
    bg = most_common_color(grid)
    seen = set()
    components = []
    for r in range(height):
        for c in range(width):
            if (r, c) in seen:
                continue
            color = grid[r][c]
            if not include_background and color == bg:
                seen.add((r, c))
                continue
            q = deque([(r, c)])
            seen.add((r, c))
            cells = []
            while q:
                cr, cc = q.popleft()
                cells.append((cr, cc))
                for nr, nc in ((cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)):
                    if 0 <= nr < height and 0 <= nc < width and (nr, nc) not in seen and grid[nr][nc] == color:
                        seen.add((nr, nc))
                        q.append((nr, nc))
            bbox = bounding_box(cells)
            components.append({
                'color': color,
                'cells': cells,
                'size': len(cells),
                'bbox': bbox,
                'grid': crop_bbox(grid, bbox),
            })
    return components


def rotate90(grid):
    return [list(row) for row in zip(*grid[::-1])]


def rotate180(grid):
    return [row[::-1] for row in grid[::-1]]


def rotate270(grid):
    return [list(row) for row in zip(*grid)][::-1]


def flip_horizontal(grid):
    return [row[::-1] for row in grid]


def flip_vertical(grid):
    return grid[::-1]


def transpose(grid):
    return [list(row) for row in zip(*grid)]


def tile_grid(grid, row_factor, col_factor):
    return [row * col_factor for _ in range(row_factor) for row in grid]


def scale_grid(grid, row_factor, col_factor):
    scaled = []
    for row in grid:
        expanded_row = []
        for value in row:
            expanded_row.extend([value] * col_factor)
        for _ in range(row_factor):
            scaled.append(expanded_row[:])
    return scaled


def apply_color_map(grid, mapping):
    return [[mapping.get(value, value) for value in row] for row in grid]



def learn_color_map_between(source_grids, target_grids, require_change=False):
    mapping = {}
    changed = False
    for src, dst in zip(source_grids, target_grids):
        if grid_shape(src) != grid_shape(dst):
            return None
        for r, row in enumerate(src):
            for c, value in enumerate(row):
                target = dst[r][c]
                if value in mapping and mapping[value] != target:
                    return None
                mapping[value] = target
                changed = changed or value != target
    if require_change and not changed:
        return None
    return mapping


def overlay_non_background(base, addition):
    if grid_shape(base) != grid_shape(addition):
        return None
    bg = most_common_color(base)
    result = copy_grid(base)
    for r, row in enumerate(addition):
        for c, value in enumerate(row):
            if value != bg:
                result[r][c] = value
    return result


def fill_enclosed_background_regions(grid):
    height, width = grid_shape(grid)
    bg = most_common_color(grid)
    seen = set()
    result = copy_grid(grid)
    for r in range(height):
        for c in range(width):
            if (r, c) in seen or grid[r][c] != bg:
                continue
            q = deque([(r, c)])
            seen.add((r, c))
            region = []
            touches_border = False
            neighbor_colors = set()
            while q:
                cr, cc = q.popleft()
                region.append((cr, cc))
                if cr in (0, height - 1) or cc in (0, width - 1):
                    touches_border = True
                for nr, nc in ((cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)):
                    if 0 <= nr < height and 0 <= nc < width:
                        if grid[nr][nc] == bg and (nr, nc) not in seen:
                            seen.add((nr, nc))
                            q.append((nr, nc))
                        elif grid[nr][nc] != bg:
                            neighbor_colors.add(grid[nr][nc])
            if not touches_border and len(neighbor_colors) == 1:
                fill_color = next(iter(neighbor_colors))
                for rr, cc in region:
                    result[rr][cc] = fill_color
    return result


def learn_color_map(train_pairs):
    return learn_color_map_between(
        [inp for inp, _ in train_pairs],
        [out for _, out in train_pairs],
        require_change=True,
    )


def find_subgrid_locations(grid, pattern):
    gh, gw = grid_shape(grid)
    ph, pw = grid_shape(pattern)
    if ph == 0 or pw == 0 or ph > gh or pw > gw:
        return []
    locations = []
    for top in range(gh - ph + 1):
        for left in range(gw - pw + 1):
            if crop_grid(grid, top, left, ph, pw) == pattern:
                locations.append((top, left))
    return locations


def fill_bounding_boxes(grid):
    result = copy_grid(grid)
    bg = most_common_color(grid)
    for color in colors_in(grid) - {bg}:
        bbox = color_bbox(grid, color)
        if bbox is None:
            continue
        top, left, bottom, right = bbox
        for r in range(top, bottom):
            for c in range(left, right):
                if result[r][c] == bg:
                    result[r][c] = color
    return result


def select_component(grid, strategy):
    comps = connected_components(grid)
    if not comps:
        return None
    if strategy == 'largest':
        comp = max(comps, key=lambda item: item['size'])
    elif strategy == 'smallest':
        comp = min(comps, key=lambda item: item['size'])
    elif strategy == 'top_left':
        comp = min(comps, key=lambda item: (item['bbox'][0], item['bbox'][1]))
    elif strategy == 'bottom_right':
        comp = max(comps, key=lambda item: (item['bbox'][2], item['bbox'][3]))
    else:
        return None
    return comp['grid']



def component_shape_signature(component):
    top, left, _, _ = component['bbox']
    return frozenset((r - top, c - left) for r, c in component['cells'])


def select_unique_component(grid, property_name):
    comps = connected_components(grid)
    if not comps:
        return None
    if property_name == 'size':
        values = [comp['size'] for comp in comps]
    elif property_name == 'color':
        values = [comp['color'] for comp in comps]
    elif property_name == 'shape':
        values = [component_shape_signature(comp) for comp in comps]
    else:
        return None
    counts = Counter(values)
    unique_indexes = [i for i, value in enumerate(values) if counts[value] == 1]
    if len(unique_indexes) != 1:
        return None
    return comps[unique_indexes[0]]['grid']


def fill_lines_between_same_color(grid):
    result = copy_grid(grid)
    bg = most_common_color(grid)
    height, width = grid_shape(grid)

    for r in range(height):
        positions_by_color = {}
        for c, value in enumerate(grid[r]):
            if value != bg:
                positions_by_color.setdefault(value, []).append(c)
        for color, cols in positions_by_color.items():
            if len(cols) >= 2:
                for c in range(min(cols), max(cols) + 1):
                    result[r][c] = color

    for c in range(width):
        positions_by_color = {}
        for r in range(height):
            value = grid[r][c]
            if value != bg:
                positions_by_color.setdefault(value, []).append(r)
        for color, rows in positions_by_color.items():
            if len(rows) >= 2:
                for r in range(min(rows), max(rows) + 1):
                    result[r][c] = color

    return result


def foreground_cells_by_color(grid):
    bg = most_common_color(grid)
    cells = []
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            if value != bg:
                cells.append((r, c, value))
    return cells


def learn_global_foreground_shift(train_pairs):
    learned_shift = None
    for inp, out in train_pairs:
        if grid_shape(inp) != grid_shape(out):
            return None
        in_cells = foreground_cells_by_color(inp)
        out_cells = foreground_cells_by_color(out)
        if not in_cells or len(in_cells) != len(out_cells):
            return None
        in_sorted = sorted(in_cells, key=lambda item: (item[2], item[0], item[1]))
        out_sorted = sorted(out_cells, key=lambda item: (item[2], item[0], item[1]))
        shifts = set()
        for (ir, ic, iv), (orow, oc, ov) in zip(in_sorted, out_sorted):
            if iv != ov:
                return None
            shifts.add((orow - ir, oc - ic))
        if len(shifts) != 1:
            return None
        shift = next(iter(shifts))
        if shift == (0, 0):
            return None
        if learned_shift is None:
            learned_shift = shift
        elif learned_shift != shift:
            return None
    return learned_shift


def shift_foreground(grid, dr, dc):
    height, width = grid_shape(grid)
    bg = most_common_color(grid)
    result = [[bg for _ in range(width)] for _ in range(height)]
    for r, c, value in foreground_cells_by_color(grid):
        nr, nc = r + dr, c + dc
        if not (0 <= nr < height and 0 <= nc < width):
            return None
        result[nr][nc] = value
    return result


def learn_tiling_rule(train_pairs):
    candidates = []
    for mode in ('tile', 'scale'):
        factors = None
        ok = True
        for inp, out in train_pairs:
            ih, iw = grid_shape(inp)
            oh, ow = grid_shape(out)
            if ih == 0 or iw == 0 or oh % ih or ow % iw:
                ok = False
                break
            current = (oh // ih, ow // iw)
            predicted = tile_grid(inp, *current) if mode == 'tile' else scale_grid(inp, *current)
            if predicted != out:
                ok = False
                break
            if factors is None:
                factors = current
            elif factors != current:
                ok = False
                break
        if ok and factors is not None and factors != (1, 1):
            candidates.append((mode, factors))
    return candidates


def candidate_rules(train_pairs):
    rules = []

    transform_rules = [
        ('identity', copy_grid),
        ('rotate90', rotate90),
        ('rotate180', rotate180),
        ('rotate270', rotate270),
        ('flip_horizontal', flip_horizontal),
        ('flip_vertical', flip_vertical),
        ('transpose', transpose),
    ]

    def add_rule(name, fn):
        if name not in {existing_name for existing_name, _ in rules}:
            rules.append((name, fn))

    outputs = [out for _, out in train_pairs]
    if outputs and all(out == outputs[0] for out in outputs):
        add_rule('constant_output', lambda grid, output=outputs[0]: copy_grid(output))

    # Similarity step: try whole-grid transformations first.
    for name, fn in transform_rules:
        transformed = [fn(inp) for inp, _ in train_pairs]
        if all(src == dst for src, dst in zip(transformed, outputs)):
            add_rule(name, fn)
        mapping = learn_color_map_between(transformed, outputs, require_change=True)
        if mapping is not None:
            add_rule(
                f'{name}_then_color_map',
                lambda grid, fn=fn, mapping=mapping: apply_color_map(fn(grid), mapping),
            )

    # Color step: if structure is unchanged, learn a direct color correspondence.
    mapping = learn_color_map(train_pairs)
    if mapping is not None:
        add_rule('color_map', lambda grid, mapping=mapping: apply_color_map(grid, mapping))

    # Difference step: if the output is smaller, try selecting or cropping meaningful objects.
    crop_selectors = []
    crop_selectors.append(('crop_non_background_bbox', lambda grid: crop_bbox(grid, non_background_bbox(grid))))

    shared_colors = set.intersection(*(colors_in(inp) for inp, _ in train_pairs)) if train_pairs else set()
    for color in sorted(shared_colors):
        crop_selectors.append((
            f'crop_color_{color}_bbox',
            lambda grid, color=color: crop_bbox(grid, color_bbox(grid, color)),
        ))

    for strategy in ('largest', 'smallest', 'top_left', 'bottom_right'):
        crop_selectors.append((
            f'select_{strategy}_component',
            lambda grid, strategy=strategy: select_component(grid, strategy),
        ))

    for property_name in ('size', 'color', 'shape'):
        crop_selectors.append((
            f'select_unique_{property_name}_component',
            lambda grid, property_name=property_name: select_unique_component(grid, property_name),
        ))

    for name, selector in crop_selectors:
        selected = [selector(inp) for inp, _ in train_pairs]
        if all(candidate is not None for candidate in selected):
            if all(candidate == out for candidate, out in zip(selected, outputs)):
                add_rule(name, selector)

            mapping = learn_color_map_between(selected, outputs, require_change=True)
            if mapping is not None:
                add_rule(
                    f'{name}_then_color_map',
                    lambda grid, selector=selector, mapping=mapping: apply_color_map(selector(grid), mapping),
                )

            # Limited rule composition: select/crop first, then apply one visual transform, then optional recolor.
            # This is intentionally small because it follows the manual order: isolate object, compare shape, map colors.
            for transform_name, transform in transform_rules[1:]:
                transformed_selected = [transform(candidate) for candidate in selected]
                if all(candidate == out for candidate, out in zip(transformed_selected, outputs)):
                    add_rule(
                        f'{name}_then_{transform_name}',
                        lambda grid, selector=selector, transform=transform: transform(selector(grid)),
                    )
                composed_mapping = learn_color_map_between(transformed_selected, outputs, require_change=True)
                if composed_mapping is not None:
                    add_rule(
                        f'{name}_then_{transform_name}_then_color_map',
                        lambda grid, selector=selector, transform=transform, mapping=composed_mapping: apply_color_map(transform(selector(grid)), mapping),
                    )

    # Difference step: if the output is larger, try repetition or pixel expansion.
    for mode, factors in learn_tiling_rule(train_pairs):
        if mode == 'tile':
            add_rule(f'tile_{factors[0]}x{factors[1]}', lambda grid, factors=factors: tile_grid(grid, *factors))
        else:
            add_rule(f'scale_{factors[0]}x{factors[1]}', lambda grid, factors=factors: scale_grid(grid, *factors))

    same_size_pairs = [grid_shape(inp) == grid_shape(out) for inp, out in train_pairs]
    if all(same_size_pairs):
        for name, fn in transform_rules[1:]:
            def overlay_rule(grid, fn=fn):
                return overlay_non_background(grid, fn(grid))
            if all(overlay_rule(inp) == out for inp, out in train_pairs):
                add_rule(f'overlay_{name}', overlay_rule)

            def reverse_overlay_rule(grid, fn=fn):
                transformed = fn(grid)
                if grid_shape(transformed) != grid_shape(grid):
                    return None
                return overlay_non_background(transformed, grid)
            if all(reverse_overlay_rule(inp) == out for inp, out in train_pairs):
                add_rule(f'overlay_input_on_{name}', reverse_overlay_rule)

        shift = learn_global_foreground_shift(train_pairs)
        if shift is not None:
            dr, dc = shift
            add_rule(f'shift_foreground_{dr}_{dc}', lambda grid, dr=dr, dc=dc: shift_foreground(grid, dr, dc))

    if all(fill_bounding_boxes(inp) == out for inp, out in train_pairs):
        add_rule('fill_bounding_boxes', fill_bounding_boxes)

    if all(fill_enclosed_background_regions(inp) == out for inp, out in train_pairs):
        add_rule('fill_enclosed_background_regions', fill_enclosed_background_regions)

    if all(fill_lines_between_same_color(inp) == out for inp, out in train_pairs):
        add_rule('fill_lines_between_same_color', fill_lines_between_same_color)

    # Last-resort memorized crop: useful when the same relative crop appears in every example.
    crop_offsets = []
    crop_ok = True
    for inp, out in train_pairs:
        locations = find_subgrid_locations(inp, out)
        if len(locations) != 1:
            crop_ok = False
            break
        crop_offsets.append((locations[0], grid_shape(out)))
    if crop_ok and len({item for item in crop_offsets}) == 1:
        (top, left), (height, width) = crop_offsets[0]
        add_rule('fixed_position_crop', lambda grid, top=top, left=left, height=height, width=width: crop_grid(grid, top, left, height, width))

    return rules


def fallback_prediction(train_pairs, test_input):
    # A valid fallback is better than a malformed answer. Use the input shape when possible.
    if test_input:
        return copy_grid(test_input)
    if train_pairs:
        return copy_grid(train_pairs[0][1])
    return [[0]]


def secondary_fallback_prediction(train_pairs, primary):
    if train_pairs and train_pairs[0][1] != primary:
        return copy_grid(train_pairs[0][1])
    return copy_grid(primary)


def solve_task(task):
    train_pairs = [(item['input'], item['output']) for item in task['train']]
    rules = candidate_rules(train_pairs)
    predictions = []
    prediction_log = []

    for test_index, test_item in enumerate(task['test']):
        test_input = test_item['input']
        task_predictions = []
        prediction_sources = []
        for rule_name, rule in rules:
            try:
                prediction = rule(test_input)
            except Exception:
                prediction = None
            if prediction and prediction not in task_predictions:
                task_predictions.append(prediction)
                prediction_sources.append(rule_name)
        if not task_predictions:
            task_predictions.append(fallback_prediction(train_pairs, test_input))
            prediction_sources.append('fallback_copy_input')
        if len(task_predictions) == 1:
            task_predictions.append(secondary_fallback_prediction(train_pairs, task_predictions[0]))
            prediction_sources.append('secondary_fallback_training_output')
        predictions.append({
            'attempt_1': task_predictions[0],
            'attempt_2': task_predictions[1],
        })
        prediction_log.append({
            'test_index': test_index,
            'attempt_1_rule': prediction_sources[0],
            'attempt_2_rule': prediction_sources[1],
            'all_matching_rules': [name for name, _ in rules],
        })

    return predictions, [name for name, _ in rules], prediction_log






In [ ]:
# Create a submission dict using the Similarity-Difference-Color rule library.
submission = {}
rule_usage = Counter()
attempt_1_usage = Counter()
solved_by_any_rule = 0
rule_log = {}

for key, task in tqdm(test_challenges.items()):
    task_predictions, matched_rules, prediction_log = solve_task(task)
    submission[key] = task_predictions
    rule_log[key] = prediction_log
    if matched_rules:
        solved_by_any_rule += 1
        for rule_name in matched_rules:
            rule_usage[rule_name] += 1
    for item in prediction_log:
        attempt_1_usage[item['attempt_1_rule']] += 1

summary = {
    'num_tasks': len(submission),
    'tasks_with_at_least_one_matching_rule': solved_by_any_rule,
    'matched_rule_counts': dict(rule_usage.most_common()),
    'attempt_1_rule_counts': dict(attempt_1_usage.most_common()),
}

# submission.json is the file the ARC evaluator reads.
with open('submission.json', 'w') as fp:
    json.dump(submission, fp)

# These two files are for Milestone 3 analysis. They are not required by Gradescope.
with open('arc_solver_rule_log.json', 'w') as fp:
    json.dump(rule_log, fp, indent=2)
with open('arc_solver_summary.json', 'w') as fp:
    json.dump(summary, fp, indent=2)

print(f'Generated predictions for {len(submission)} tasks.')
print(f'Tasks with at least one matching rule: {solved_by_any_rule}')
print('Most common matched rules:', rule_usage.most_common(10))
print('Most common attempt_1 rules:', attempt_1_usage.most_common(10))



Here is what our submission for the test task above looks like:

In [ ]:
print('Sample submission entry:')
submission[sample_task]


# Confused about where to get started? 

If you're not sure what an initial solution might look like, then consider looking at public notebooks here: https://www.kaggle.com/competitions/arc-prize-2024/code or https://www.kaggle.com/competitions/arc-prize-2025/code or joining the public discussion here: https://www.kaggle.com/competitions/arc-prize-2024/discussion and https://www.kaggle.com/competitions/arc-prize-2025/discussion.

One example notebook that uses a very simple knowledge-based approach is this one: https://www.kaggle.com/code/michaelhodel/program-synthesis-starter-notebook/notebook, which conducts search over a space of domain specific block languages to form hypotheses and then applies these to test items.